# Installation & First Graph

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will install LangGraph, set up your API key, and build a minimal graph that echoes input through a single node.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Build a Minimal Echo Graph

A single-node graph that takes an input message and echoes it back.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    message: str

def echo(state: State) -> dict:
    return {"message": f"Echo: {state['message']}"}

graph = StateGraph(State)
graph.add_node("echo", echo)
graph.add_edge(START, "echo")
graph.add_edge("echo", END)

app = graph.compile()

result = app.invoke({"message": "Hello, LangGraph!"})
print(result["message"])

## 4. A Two-Node Graph with an LLM

Extend to two nodes: generate a joke, then rate it.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

class JokeState(TypedDict):
    topic: str
    joke: str
    rating: str

def generate_joke(state: JokeState) -> dict:
    response = llm.invoke(f"Tell a short joke about: {state['topic']}")
    return {"joke": response.content}

def rate_joke(state: JokeState) -> dict:
    response = llm.invoke(
        f"Rate this joke from 1-10 and explain why in one sentence:\n{state['joke']}"
    )
    return {"rating": response.content}

graph = StateGraph(JokeState)

graph.add_node("generate", generate_joke)
graph.add_node("rate", rate_joke)

graph.add_edge(START, "generate")
graph.add_edge("generate", "rate")
graph.add_edge("rate", END)

app = graph.compile()

result = app.invoke({"topic": "AI agents"})
print(f"Joke: {result['joke']}")
print(f"Rating: {result['rating']}")

## 5. Visualize the Graph

In [ ]:
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

## What You Learned

1. **Install** LangGraph with `pip install langgraph langchain`
2. **State** is a `TypedDict` that flows through every node
3. **Nodes** are functions that receive state and return partial updates
4. **Edges** connect nodes: `START → node → END`
5. **`graph.compile()`** produces a runnable app you call with `.invoke()`